# 00 — Toy version

**Do this one first. One afternoon.**

Three patients I type by hand. Whole policy pasted into the prompt. No chunking, no retrieval,
no embeddings.

The goal is to watch the model hand me a confident quote that isn't in the document. Once I've
seen that happen, the rest of the project has a reason to exist.

Bonus: this is secretly row 0 of my results table.


In [1]:
from pathlib import Path
import os

os.chdir("/content/pa-appeal")

print("Current folder:", Path.cwd())
print("Git repository:", (Path(".git")).exists())
print("Toy notebook:", Path("notebooks/00_toy.ipynb").exists())
print("Requirements:", Path("requirements.txt").exists())

Current folder: /content/pa-appeal
Git repository: True
Toy notebook: True
Requirements: True


In [2]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


working from: /content/pa-appeal


In [3]:
from getpass import getpass
import os

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")

assert os.environ.get("GEMINI_API_KEY"), "No API key found"
print("Key loaded")

Key loaded


In [4]:

from typing import Literal

from google import genai
from pydantic import BaseModel


class Decision(BaseModel):
    criterion_id: str
    label: Literal[
        "met",
        "unmet",
        "insufficient_evidence",
    ]
    evidence_quote: str
    reasoning: str


class DecisionResponse(BaseModel):
    decisions: list[Decision]


client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

MODEL = "gemini-3.1-flash-lite"

def ask_json(prompt, system):
    interaction = client.interactions.create(
        model=MODEL,
        input=f"""{system}

{prompt}""".strip(),
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": DecisionResponse.model_json_schema(),
        },
    )

    parsed = DecisionResponse.model_validate_json(
        interaction.output_text
    )

    return parsed.model_dump()


print("Gemini client ready")


Gemini client ready


## The policy\n\nDownload L33718 by hand first, save it as `data/policies/L33718.md`.

In [5]:
policy = Path("data/policies/NCD240.4.md").read_text()
print(len(policy), "characters")
print(policy[:600])


13546 characters
# Continuous Positive Airway Pressure (CPAP) Therapy For Obstructive Sleep Apnea (OSA)

## Tracking Information

Publication Number  
100-3

Manual Section Number  
240.4

Manual Section Title  
Continuous Positive Airway Pressure (CPAP) Therapy For Obstructive Sleep Apnea (OSA)

Version Number

Effective Date of this Version  
03/13/2008

Ending Effective Date of this Version

Implementation Date  
08/04/2008

Implementation QR Modifier Date

## Description Information

Benefit Category  
Durable Medical Equipment  

**Please Note:** This may not be an exhaustive list of all applicable Medica


In [6]:
policy = Path("data/policies/L33718.md").read_text(encoding="utf-8")

checks = {
    "title": policy.startswith("# Positive Airway Pressure"),
    "apnea definition": "Apnea is defined as the cessation of airflow for at least 10 seconds." in policy,
    "AHI 15 rule": "greater than or equal to 15 events per hour with a minimum of 30 events" in policy,
    "reevaluation window": "no sooner than the 31st day but no later than the 91st day" in policy,
    "adherence rule": "Adherence to therapy is defined as use of PAP" in policy,
    "E0471 rule": "E0471) is not reasonable and necessary if the primary diagnosis is OSA" in policy,
    "ending sections": "## Associated Documents" in policy,
    "CSS removed": ".svg-inline--fa" not in policy,
}

for name, passed in checks.items():
    print(f"{name:24} {passed}")

assert all(checks.values()), "Policy failed at least one completeness check"
print("L33718 looks complete.")

title                    True
apnea definition         True
AHI 15 rule              True
reevaluation window      True
adherence rule           True
E0471 rule               True
ending sections          True
CSS removed              True
L33718 looks complete.


## Three patients, typed by hand

One obvious yes, one obvious no, one where the record just doesn't say.

In [7]:
cases = [
    {"id": "yes", "record": """
Sleep study, 03/14: AHI 24.6 events/hour. Total respiratory events: 142.
Patient reports falling asleep at his desk most afternoons.
Ordered: E0601 CPAP.
""".strip()},

    {"id": "no", "record": """
Sleep study, 03/14: AHI 3.1 events/hour. Total respiratory events: 19.
No daytime sleepiness. No hypertension. No cardiac history.
Ordered: E0471 bi-level with backup rate. Primary diagnosis: obstructive sleep apnea.
""".strip()},

    {"id": "silent", "record": """
Sleep study, 03/14: study completed, patient tolerated well.
Follow-up scheduled. Ordered: E0601 CPAP.
""".strip()},
]

for c in cases:
    print(c["id"], "-", len(c["record"]), "chars")


yes - 153 chars
no - 216 chars
silent - 102 chars


## Ask

Two rules only for now. Note the line about absent information — leave it out and the model
labels a silent record "unmet" almost every time.

In [8]:
SYSTEM = """You decide whether a patient record satisfies Medicare coverage criteria.

Rules:
1. Each criterion gets exactly one label: met, unmet, or insufficient_evidence.
2. ABSENT INFORMATION IS insufficient_evidence, NOT unmet. If the record is silent on a
   value, or only implies it, the label is insufficient_evidence.
3. evidence_quote must be copied character-for-character from the policy text below.
   Do not paraphrase, do not tidy punctuation, do not write it from memory.
4. If you cannot find a supporting quote in the policy text, use insufficient_evidence
   and leave evidence_quote empty.

Return JSON: {"decisions": [{"criterion_id", "label", "evidence_quote", "reasoning"}]}
"""

CRITERIA_ASKED = """
B1: sleep study shows AHI or RDI of at least 15 events per hour, with at least 30 events
E0471_osa: E0471 (bi-level with backup rate) is not covered when the primary diagnosis is OSA
"""

def run_case(record):
    prompt = f"POLICY TEXT:\n{policy}\n\nCRITERIA:\n{CRITERIA_ASKED}\n\nPATIENT RECORD:\n{record}"
    return ask_json(prompt, SYSTEM)

# out = {c["id"]: run_case(c["record"]) for c in cases}
# out["yes"]
out = globals().get("out", {})

for case in cases:
    case_id = case["id"]

    if case_id in out:
        print(f"Skipping completed case: {case_id}")
        continue

    print(f"Running case: {case_id}")
    out[case_id] = run_case(case["record"])
    print(f"Completed: {case_id}")

out["yes"]


Running case: yes
Completed: yes
Running case: no
Completed: no
Running case: silent
Completed: silent


{'decisions': [{'criterion_id': 'B1',
   'label': 'met',
   'evidence_quote': 'The apnea-hypopnea index (AHI) or Respiratory Disturbance Index (RDI) is greater than or equal to 15 events per hour with a minimum of 30 events',
   'reasoning': "The patient's sleep study demonstrates an AHI of 24.6 events/hour with a total of 142 events, satisfying both the frequency and minimum event count requirements."},
  {'criterion_id': 'E0471_osa',
   'label': 'met',
   'evidence_quote': 'A bi-level positive airway pressure device with back-up rate (E0471) is not reasonable and necessary if the primary diagnosis is OSA. If an E0471 is billed with a diagnosis of OSA, it will be denied as not reasonable and necessary.',
   'reasoning': 'The patient has been ordered an E0601 (CPAP), not an E0471. Since the order is not for E0471, the restriction against using E0471 for OSA is not violated.'}]}

## The whole point of this notebook

Check every quote. `in` is the entire verification method.

In [9]:
for case_id, result in out.items():
    print("=" * 60)
    print(case_id)
    for d in result["decisions"]:
        q = d["evidence_quote"]
        found = q.strip() != "" and q in policy
        print(f"  {d['criterion_id']:<12} {d['label']:<22} quote in policy: {found}")
        if q and not found:
            print(f"    >>> {q[:120]}")


yes
  B1           met                    quote in policy: True
  E0471_osa    met                    quote in policy: True
no
  B1           unmet                  quote in policy: True
  E0471_osa    met                    quote in policy: True
silent
  B1           insufficient_evidence  quote in policy: False
  E0471_osa    met                    quote in policy: True


## What I should see

- The obvious-yes case labelled met, the obvious-no case labelled unmet.
- The silent case labelled insufficient_evidence (if it says unmet instead, rule 2 needs
  more emphasis).
- **At least one quote coming back `False`.** Either invented outright, or subtly reworded.

Some `False` results are my fault, not the model's — curly quotes and line breaks. That's what
`normalize()` in notebook 03 fixes. The ones left after normalizing are the real thing.

Once I've seen this, go to notebook 01.